# 单通道：人工挑选 trace + 池化

`smTest_batch` 已经把每个视野的**完整 trace 库**写成 `<目录>/traces.npz`。这一本把它们
逐个摊开、勾选要留下的，再把选中的池化到一处，交给下一步处理。

**与 `dualCh/smTest_dual_manual.ipynb` 的区别**（别照搬那边的习惯）：

| | 双通道版 | 这一本 |
|---|---|---|
| 勾选含义 | 勾上 = **剔除** | 勾上 = **保留**（选中） |
| 输入 | `fret.csv` + `good_traces.csv`（已粗筛） | `traces.npz`（**未筛选**，全部分子） |
| 保存 | 留着和不留的都画图 | **只存选中的**，不要的不写 |
| 池化 | E 分布 | 池化 **trace 数据本身**（给下一步用） |

每个视野写到 `<目录>/`（`SUFFIX` 可改）：

| 文件 | 内容 |
|---|---|
| `traces_manual.npz` | **只含选中的** trace（键与 `traces.npz` 相同） |
| `emitters_manual.csv` | 对应的 `emitters.csv` 行子集 |
| `traces_manual.png` | 选中的那些一屏看完（按亮度排序的热图 + 群体中位） |
| `manual_select.json` | 选了哪些编号 —— 重启内核后自动读回，不会把选过的丢掉 |

池化结果写到 `POOL_DIR/<组名>/`：`traces_pooled.npz` + `pooled_index.csv`。


In [ ]:
# `import smlm` 需要 Py_Smile 在 sys.path 上; notebook 放在子目录里时自己找一下
import json
import pathlib
import sys

for _q in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_q / 'smlm' / '__init__.py').is_file():
        sys.path.insert(0, str(_q))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ======================= 输入 =======================
BASE_DIR = pathlib.Path(r"G:\SPARK\20261124")   # 底下是各个 <主名>_trace/ 目录
FIELDS = []            # 空 = BASE_DIR 下所有 *trace*/ 目录; 也可填关键字, 如 ['fov3', 'fov4']
SUFFIX = '_manual'     # 输出后缀

# ======================= 翻页 =======================
PER_PAGE = 48          # 每页画多少条 (8 列 x 6 行)
SORT = 'brightness'    # 'brightness' = 按 95% 分位亮度降序 (先看亮的); 'index' = 按编号

# ======================= 池化 =======================
# 组名 -> 目录名关键字列表 (空 = 该组用全部); 每组写到 POOL_DIR/<组名>/
GROUPS = {
    'all': [],
    # 'fov3_fov4': ['fov3', 'fov4'],
}
POOL_DIR = BASE_DIR / 'TRACE_pooled'

# 一般不用手填: 勾选结果存进各目录的 manual_select.json, 重跑时自动读回
SELECT = {}

print('BASE_DIR :', BASE_DIR)
print('视野目录 :')
for p in sorted(BASE_DIR.glob('*trace*')):
    if p.is_dir() and (p / 'traces.npz').is_file():
        print('   ', p.name)
print('池化输出 :', POOL_DIR)


## 读回

`traces.npz` 里的键：`emitter_id / x_px / y_px / intensity(N,T) / bg_flux(N,T) / t_s`
+ 标量 `sigma_px / exposure_s / n_frames / source`。

各文件帧数**可能不同**（1000 / 2000 / 4000），所以每条 trace 按自己的长度处理；
池化时才补 NaN 对齐。


In [ ]:
SELECT_FILE = 'manual_select.json'


def field_dirs(keys=None, *, need_npz=True):
    """BASE_DIR 下的 trace 目录; keys 为空 = 全部。匹配不到就报错。"""
    all_ = [p for p in sorted(BASE_DIR.glob('*trace*'))
            if p.is_dir() and (not need_npz or (p / 'traces.npz').is_file())]
    if keys is None or not keys:
        return all_
    out = [p for p in all_ if any(k in p.name for k in keys)]
    if not out:
        raise ValueError(f'没有目录匹配 {keys}; 现有 {[p.name for p in all_]}')
    return out


def load_traces(d):
    """读一个目录的 traces.npz -> dict (标量转成 python 值)。"""
    z = np.load(d / 'traces.npz')
    out = {k: z[k] for k in z.files}
    z.close()
    for k in ('sigma_px', 'exposure_s'):
        if k in out:
            out[k] = float(out[k])
    out['n_frames'] = int(out['n_frames']) if 'n_frames' in out else int(
        out['intensity'].shape[1])
    return out


def order_traces(data):
    """要显示的顺序 (亮度降序让好看的先出现)。"""
    I = np.asarray(data['intensity'], float)
    if SORT == 'brightness':
        return np.argsort(-np.nanpercentile(I, 95, axis=1))
    return np.arange(len(data['emitter_id']))


def read_select(d):
    """该目录已选中的编号; 没有 json 时返回 None (= 还没挑过, 默认全选)。"""
    p = d / SELECT_FILE
    if not p.is_file():
        return None
    try:
        return [int(v) for v in json.loads(
            p.read_text(encoding='utf-8')).get('selected', [])]
    except (ValueError, TypeError):
        print(f'  [!] {p} 读不动, 当成没挑过')
        return None


def write_select(d, ids):
    (d / SELECT_FILE).write_text(
        json.dumps({'selected': sorted(int(i) for i in ids)},
                   ensure_ascii=False, indent=1), encoding='utf-8')


def save_selected(d, ids):
    """只把选中的写到 traces_manual.npz / emitters_manual.csv / traces_manual.png。"""
    data = load_traces(d)
    keep = np.isin(data['emitter_id'], sorted(int(i) for i in ids))
    idx = np.flatnonzero(keep)
    if idx.size == 0:
        print(f'  [!] {d.name}: 一条都没选, 跳过')
        return 0

    sub = {k: (v[idx] if isinstance(v, np.ndarray) and v.ndim >= 1
               and v.shape[0] == keep.size else v)
           for k, v in data.items()}
    np.savez_compressed(d / f'traces{SUFFIX}.npz', **sub)

    em = pd.read_csv(d / 'emitters.csv', encoding='utf-8-sig')
    em[em.idx.isin(sub['emitter_id'])].to_csv(
        d / f'emitters{SUFFIX}.csv', index=False, encoding='utf-8-sig')

    _heatmap_fig(np.asarray(sub['intensity'], float),
                 np.asarray(sub['t_s'], float),
                 f'{d.name}{SUFFIX}   n={idx.size}').savefig(
        d / f'traces{SUFFIX}.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close('all')
    write_select(d, sub['emitter_id'])
    return int(idx.size)


def _heatmap_fig(I, t_s, title):
    """全部 trace 一屏: 按亮度排序的热图 + 群体中位。"""
    if I.size == 0:
        raise ValueError('没有数据')
    order = np.argsort(-np.nanpercentile(I, 95, axis=1))
    I = I[order]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0),
                             gridspec_kw=dict(width_ratios=[3, 1]))
    im = axes[0].imshow(I, aspect='auto', cmap='magma',
                        extent=[t_s[0], t_s[-1], len(I), 0],
                        vmin=0.0, vmax=float(np.nanpercentile(I, 99)))
    axes[0].set_xlabel('time (s)', fontsize=8)
    axes[0].set_ylabel(f'trace (sorted by brightness, n={len(I)})', fontsize=8)
    axes[0].tick_params(labelsize=7)
    fig.colorbar(im, ax=axes[0], pad=0.01, label='photons / frame')
    axes[1].plot(t_s, np.nanmedian(I, axis=0), color='0.2', lw=0.8)
    axes[1].set_title('population median', fontsize=8)
    axes[1].set_xlabel('time (s)', fontsize=8)
    axes[1].tick_params(labelsize=7)
    axes[1].grid(alpha=0.25)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return fig


def _grid_fig(I, t_s, ids, *, ncols=8, dpi=110, title=''):
    """一页的逐条曲线。"""
    nrow = int(np.ceil(len(ids) / ncols))
    fig, axes = plt.subplots(nrow, ncols, figsize=(1.45 * ncols, 1.2 * nrow),
                             dpi=dpi, squeeze=False, sharex=True)
    for k, ax in enumerate(axes.ravel()):
        if k >= len(ids):
            ax.axis('off')
            continue
        ax.plot(t_s, I[k], lw=0.6, color='tab:blue')
        ax.axhline(0.0, color='0.85', lw=0.4)
        ax.set_title(f'#{ids[k]}', fontsize=5.5, pad=1.5)
        ax.tick_params(labelsize=4.5, length=1.5, pad=1)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
    if title:
        fig.suptitle(title, fontsize=8, y=1.0)
    fig.tight_layout(pad=0.3)
    return fig


## 一页一段：**选中 = 保留**

每页是某个视野里的 `PER_PAGE` 条 trace（默认按亮度降序，好看的先出现）。图上画的是
**这一页的全部**曲线，勾选框表示"要留下"；两面互相对照着看。

- **换页 / 点"保存本页"会自动落盘**：把该页所属视野里**已勾选的**写成 `traces_manual.npz`
  （不要的不写），同时更新 `manual_select.json`；
- **没勾过的视野默认全选** —— 第一次跑就是"全都要"，取消掉不要的即可；
- 状态行会写"已保存 / 未保存"；`全部保存` 会把所有视野按当前勾选重写一遍。


In [ ]:
import ipywidgets as widgets
from IPython.display import display

_DIRS = field_dirs(FIELDS)
_data = {}                    # {目录名: load_traces(dict)}
_order = {}                   # {目录名: np.argsort 结果}
_sel = {}                     # {目录名: set(ids)}  —— 勾上的
_dirty = set()                # 有改动还没落盘的目录

PAGES = []                    # [(dir, a, b)] 一段 = 一个视野的一页
for _d in _DIRS:
    _n = len(load_traces(_d)['emitter_id'])
    for _a in range(0, _n, PER_PAGE):
        PAGES.append((_d, _a, min(_a + PER_PAGE, _n)))

CURRENT = {'i': 0}
_title = widgets.HTML()
_fig = widgets.Output()
_grid = widgets.HBox([])
_status = widgets.HTML()
_prev = widgets.Button(description='← 上一页', layout=widgets.Layout(width='105px'))
_next = widgets.Button(description='下一页 →', layout=widgets.Layout(width='105px'))
_all = widgets.Button(description='全选本页', layout=widgets.Layout(width='105px'))
_none = widgets.Button(description='全不选本页', layout=widgets.Layout(width='105px'))
_save = widgets.Button(description='保存本页', button_style='primary',
                       layout=widgets.Layout(width='105px'))
_save_all = widgets.Button(description='全部保存', layout=widgets.Layout(width='105px'))
_bar = widgets.HBox([_prev, _next, _all, _none, _save, _save_all])


def _get(d):
    """目录数据 (含排序与默认全选)。"""
    if d not in _data:
        _data[d] = load_traces(d)
        _order[d] = order_traces(_data[d])
        st = read_select(d)
        _sel[d] = (set(int(v) for v in st) if st is not None
                   else set(int(v) for v in _data[d]['emitter_id']))
    return _data[d]


def _page_ids(d, a, b):
    data = _get(d)
    o = _order[d][a:b]
    return o, np.asarray(data['emitter_id'])[o], np.asarray(data['intensity'])[o]


def _draw():
    d, a, b = PAGES[CURRENT['i']]
    data = _get(d)
    o, ids, I = _page_ids(d, a, b)
    t_s = np.asarray(data['t_s'], float)[:I.shape[1]]
    with _fig:
        _fig.clear_output(wait=True)
        fig = _grid_fig(I, t_s, ids, title=f'{d.name}   #{ids[0]}~#{ids[-1]}'
                                            f'   ({ids.size} 条)')
        display(fig)
    boxes = {}
    for k, i in enumerate(ids):
        peak = float(np.nanpercentile(I[k], 95))
        boxes[int(i)] = widgets.Checkbox(
            value=int(i) in _sel[d],
            description=f'#{int(i)}  {peak:.0f}',
            indent=False, layout=widgets.Layout(width='120px'))
        boxes[int(i)].observe(
            lambda ch, _i=int(i), _d=d: _toggle(_d, _i, ch['new']), names='value')
    _grid.children = tuple(boxes.values())
    _title.value = (f'<b>{CURRENT["i"] + 1} / {len(PAGES)}</b>　{d.name}　'
                    f'条 {a}–{b - 1} / {len(data["emitter_id"])}')
    _sync()


def _toggle(d, i, on):
    (_sel[d].add if on else _sel[d].discard)(int(i))
    _dirty.add(d)
    _sync()


def _sync():
    d = PAGES[CURRENT['i']][0]
    n_all = len(_get(d)['emitter_id'])
    flag = ('<span style="color:#b45309">未保存</span>' if d in _dirty
            else '<span style="color:#15803d">已保存</span>')
    _status.value = (f'{d.name}：选中 {len(_sel[d])} / {n_all} 条　{flag}')


def _save_one(d):
    n = save_selected(d, _sel[d])
    _dirty.discard(d)
    print(f'  {d.name}: 落盘 {n} 条 -> traces{SUFFIX}.npz')
    return n


def _goto(step):
    d = PAGES[CURRENT['i']][0]
    if d in _dirty:
        _save_one(d)                      # 换页前自动存
    CURRENT['i'] = (CURRENT['i'] + step) % len(PAGES)
    _draw()


def _set_page(on):
    d, a, b = PAGES[CURRENT['i']]
    _, ids, _I = _page_ids(d, a, b)
    for i in ids:
        (_sel[d].add if on else _sel[d].discard)(int(i))
    _dirty.add(d)
    _draw()


def _save_current(_btn=None):
    _save_one(PAGES[CURRENT['i']][0])
    _sync()


def _save_everything(_btn=None):
    for k, d in enumerate(dict.fromkeys(p[0] for p in PAGES), 1):
        print(f'[{k}]', end='')
        _save_one(d)
    _draw()


_prev.on_click(lambda _b: _goto(-1))
_next.on_click(lambda _b: _goto(1))
_all.on_click(lambda _b: _set_page(True))
_none.on_click(lambda _b: _set_page(False))
_save.on_click(_save_current)
_save_all.on_click(_save_everything)

_draw()
display(widgets.VBox([_bar, _title, _status, _fig, _grid]))


## 池化选中的 trace

读各目录的 `traces_manual.npz`（**不是**内存里的勾选状态），所以"池化 = 落盘的事实"。
组的划分由 `GROUPS` 决定，写到 `POOL_DIR/<组名>/`：

| 文件 | 内容 |
|---|---|
| `traces_pooled.npz` | `intensity(N,Tmax)` / `bg_flux` 按 **NaN 补到她**；真实的每 trace 长度在 `n_frames`；`t_s` 是公共时间轴 |
| `pooled_index.csv` | 行 ↔ (来源目录, emitter_id, x, y, n_frames)，下一步要回溯就靠它 |

`t_s` 要求各文件曝光时间一致（都是 0.1 s）；不一致会打印警告，仍按第一条的时间轴写。


In [ ]:
for gname, keys in GROUPS.items():
    dirs = field_dirs(keys, need_npz=False)
    have = [d for d in dirs if (d / f'traces{SUFFIX}.npz').is_file()]
    print(f'\n=== {gname} ===')
    missing = [d.name for d in dirs if d not in have]
    if missing:
        print('  [!] 还没保存过, 跳过:', ', '.join(missing))
    if not have:
        print('  空')
        continue

    blocks = []                       # [(name, I, B, ids, dt)]
    for d in have:
        z = np.load(d / f'traces{SUFFIX}.npz')
        I = np.asarray(z['intensity'], np.float32)
        B = np.asarray(z['bg_flux'], np.float32) if 'bg_flux' in z \
            else np.full_like(I, np.nan)
        ids = np.asarray(z['emitter_id'], int)
        ts = np.asarray(z['t_s'], float)
        ex = float(z['exposure_s']) if 'exposure_s' in z else np.nan
        z.close()
        dt = float(ts[1] - ts[0]) if ts.size > 1 else np.nan
        if blocks and not np.isclose(dt, blocks[0][4]):
            print(f'  [!] {d.name}: 曝光 {dt:.4f}s 与第一条 {blocks[0][4]:.4f}s 不同')
        blocks.append((d.name, I, B, ids, dt))
        print(f'  {d.name:<44} {I.shape[0]:>4} 条 x {I.shape[1]:>5} 帧')

    # 帧数不一致: 补 NaN 对齐到最长, 真实长度记在 n_frames
    T = max(b[1].shape[1] for b in blocks)
    n_tot = sum(b[1].shape[0] for b in blocks)
    Ipad = np.full((n_tot, T), np.nan, np.float32)
    Bpad = np.full((n_tot, T), np.nan, np.float32)
    idx = np.empty(n_tot, int)
    nf = np.empty(n_tot, int)
    fid = np.empty(n_tot, int)
    files = []
    off = 0
    for k, (name, I, B, ids, _dt) in enumerate(blocks):
        n = I.shape[0]
        Ipad[off:off + n, :I.shape[1]] = I
        Bpad[off:off + n, :B.shape[1]] = B
        idx[off:off + n] = ids
        nf[off:off + n] = I.shape[1]
        fid[off:off + n] = k
        files += [name] * n
        off += n

    out = POOL_DIR / gname
    out.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out / 'traces_pooled.npz',
                        intensity=Ipad, bg_flux=Bpad, emitter_id=idx,
                        file_id=fid, files=np.array([b[0] for b in blocks]),
                        n_frames=nf, t_s=np.arange(T) * blocks[0][4],
                        exposure_s=blocks[0][4])
    pd.DataFrame({'row': np.arange(n_tot), 'file': files, 'emitter_id': idx,
                  'n_frames': nf}).to_csv(
        out / 'pooled_index.csv', index=False, encoding='utf-8-sig')
    print(f'  -> {out}  共 {n_tot} 条, 最长 {T} 帧 '
          f'(帧数 {nf.min()}~{nf.max()})')
